# AI 기본법 계층형 청킹: 조 → 항 → 호 → 목

법률 구조를 다음 계층으로 저장합니다.

```text
article(조)
└── paragraph(항)
    └── item(호)
        └── subitem(목)
```

각 청크에는 `node_id`, `parent_id`, `article_id`를 저장합니다.

- `content`: 해당 단계의 직접 내용만 포함한 검색·임베딩용 텍스트
- `context_content`: 해당 단계와 모든 하위 내용을 포함한 답변용 텍스트

예를 들어 제2조제4호의 검색 벡터에는 정의 문장만 들어가고, 답변 Context에는 가목부터 카목까지 포함됩니다.


In [1]:
from pathlib import Path
import sys
import re
import uuid
import xml.etree.ElementTree as ET
from collections import Counter

from dotenv import load_dotenv
from qdrant_client.models import Distance, PointStruct, VectorParams

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "common" / "ai_model.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("프로젝트 폴더에서 실행하세요.")

sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env", override=True)

from common.ai_model import get_embedding_model
from common.qdrant import get_qdrant_client

XML_PATH = PROJECT_ROOT / "data" / "ai_basic_law" / "ai_basic_law.xml"
if not XML_PATH.exists():
    raise FileNotFoundError(f"XML 파일이 없습니다: {XML_PATH}")

root = ET.parse(XML_PATH).getroot()
for element in root.iter():
    element.tag = element.tag.rsplit("}", 1)[-1]

print("XML 로드:", XML_PATH)


XML 로드: /home/aiuser/rag_mini_pjt_template/data/ai_basic_law/ai_basic_law.xml


## 1. 계층형 청크 생성


In [2]:
SECTION_PATTERN = re.compile(r"^제\s*\d+\s*[편장절관]\s*")

def normalize(text):
    return " ".join((text or "").split())

def direct_text(node, tag):
    return normalize(node.findtext(f"./{tag}", ""))

def descendant_content(node, content_tags):
    parts = []
    for element in node.iter():
        if element.tag not in content_tags:
            continue
        text = normalize("".join(element.itertext()))
        if text and text not in parts:
            parts.append(text)
    return "\n".join(parts)

def stable_id(*parts):
    key = "/".join(str(part) for part in parts)
    return str(uuid.uuid5(uuid.NAMESPACE_URL, key))

def article_label(unit):
    number = normalize(unit.findtext("조문번호", ""))
    branch = normalize(unit.findtext("조문가지번호", ""))
    label = f"제{number}조"
    if branch and branch != "0":
        label += f"의{branch}"
    return label

law_name = normalize(root.findtext("./기본정보/법령명_한글", ""))
effective_date = normalize(root.findtext("./기본정보/시행일자", ""))
chunks = []

def add_node(
    node_type,
    node_id,
    parent_id,
    article_id,
    article,
    title,
    content,
    context_content,
    paragraph="",
    item="",
    subitem="",
):
    # 직접 내용이 없는 컨테이너는 벡터 검색 대상에서 제외하되,
    # 자식의 parent_id에는 논리적 ID를 그대로 유지합니다.
    if not content:
        return

    chunks.append({
        "id": node_id,
        "node_id": node_id,
        "parent_id": parent_id,
        "article_id": article_id,
        "chunk_type": node_type,
        "article": article,
        "paragraph": paragraph,
        "item": item,
        "subitem": subitem,
        "title": title,
        "content": content,
        "context_content": context_content or content,
        "law_name": law_name,
        "effective_date": effective_date,
        "source": XML_PATH.name,
    })

for unit in root.findall(".//조문단위"):
    article = article_label(unit)
    title = normalize(unit.findtext("조문제목", ""))
    article_direct = direct_text(unit, "조문내용")

    if not title and SECTION_PATTERN.match(article_direct):
        continue

    article_id = stable_id(law_name, effective_date, article, "article")
    article_context = descendant_content(
        unit,
        {"조문내용", "항내용", "호내용", "목내용"},
    )
    add_node(
        node_type="article",
        node_id=article_id,
        parent_id=None,
        article_id=article_id,
        article=article,
        title=title,
        content=article_direct,
        context_content=article_context,
    )

    for paragraph_index, paragraph_node in enumerate(unit.findall("./항"), start=1):
        paragraph_number = normalize(paragraph_node.findtext("항번호", ""))
        paragraph = paragraph_number or f"{paragraph_index}항"
        paragraph_id = stable_id(article_id, "paragraph", paragraph_index)

        paragraph_direct = direct_text(paragraph_node, "항내용")
        paragraph_context = descendant_content(
            paragraph_node,
            {"항내용", "호내용", "목내용"},
        )
        add_node(
            node_type="paragraph",
            node_id=paragraph_id,
            parent_id=article_id,
            article_id=article_id,
            article=article,
            title=title,
            paragraph=paragraph,
            content=paragraph_direct,
            context_content=paragraph_context,
        )

        for item_index, item_node in enumerate(paragraph_node.findall("./호"), start=1):
            item_number = normalize(item_node.findtext("호번호", ""))
            item = item_number or f"{item_index}호"
            item_id = stable_id(paragraph_id, "item", item_index)

            # 검색용 content에는 호내용만, 답변용 context에는 목까지 포함
            item_direct = direct_text(item_node, "호내용")
            item_context = descendant_content(
                item_node,
                {"호내용", "목내용"},
            )
            add_node(
                node_type="item",
                node_id=item_id,
                parent_id=paragraph_id,
                article_id=article_id,
                article=article,
                title=title,
                paragraph=paragraph,
                item=item,
                content=item_direct,
                context_content=item_context,
            )

            for subitem_index, subitem_node in enumerate(item_node.findall("./목"), start=1):
                subitem_number = normalize(subitem_node.findtext("목번호", ""))
                subitem = subitem_number or f"{subitem_index}목"
                subitem_id = stable_id(item_id, "subitem", subitem_index)
                subitem_direct = direct_text(subitem_node, "목내용")

                add_node(
                    node_type="subitem",
                    node_id=subitem_id,
                    parent_id=item_id,
                    article_id=article_id,
                    article=article,
                    title=title,
                    paragraph=paragraph,
                    item=item,
                    subitem=subitem,
                    content=subitem_direct,
                    context_content=subitem_direct,
                )

assert chunks, "생성된 청크가 없습니다."
assert len(chunks) == len({chunk["id"] for chunk in chunks}), "ID가 중복되었습니다."

print("전체 검색 청크:", len(chunks))
print("유형별:", dict(Counter(chunk["chunk_type"] for chunk in chunks)))


전체 검색 청크: 414
유형별: {'article': 46, 'item': 191, 'subitem': 13, 'paragraph': 164}


## 2. 제2조제4호 검색 텍스트와 답변 Context 확인


In [3]:
definition_chunks = [
    chunk
    for chunk in chunks
    if chunk["article"] == "제2조"
    and chunk["chunk_type"] == "item"
    and '"고영향 인공지능"이란' in chunk["content"]
]

assert len(definition_chunks) == 1, "제2조제4호 정의 청크를 하나 찾지 못했습니다."

definition = definition_chunks[0]
print("[검색·임베딩용 content]")
print(definition["content"])
print("\n[답변용 context_content]")
print(definition["context_content"])
print("\n계층:")
print("node_id:", definition["node_id"])
print("parent_id:", definition["parent_id"])
print("article_id:", definition["article_id"])


[검색·임베딩용 content]
4. "고영향 인공지능"이란 사람의 생명, 신체의 안전 및 기본권에 중대한 영향을 미치거나 위험을 초래할 우려가 있는 인공지능시스템으로서 다음 각 목의 어느 하나의 영역에서 활용되는 것을 말한다.

[답변용 context_content]
4. "고영향 인공지능"이란 사람의 생명, 신체의 안전 및 기본권에 중대한 영향을 미치거나 위험을 초래할 우려가 있는 인공지능시스템으로서 다음 각 목의 어느 하나의 영역에서 활용되는 것을 말한다.
가. 「에너지법」 제2조제1호에 따른 에너지의 공급
나. 「먹는물관리법」 제3조제1호에 따른 먹는물의 생산 공정
다. 「보건의료기본법」 제3조제1호에 따른 보건의료의 제공 및 이용체계의 구축ㆍ운영
라. 「의료기기법」 제2조제1항에 따른 의료기기 및 「디지털의료제품법」 제2조제2호에 따른 디지털의료기기의 개발 및 이용
마. 「원자력시설 등의 방호 및 방사능 방재 대책법」 제2조제1항제1호에 따른 핵물질과 같은 항 제2호에 따른 원자력시설의 안전한 관리 및 운영
바. 범죄 수사나 체포 업무를 위한 생체인식정보(얼굴ㆍ지문ㆍ홍채 및 손바닥 정맥 등 개인을 식별할 수 있는 신체적ㆍ생리적ㆍ행동적 특징에 관한 개인정보를 말한다)의 분석ㆍ활용
사. 채용, 대출 심사 등 개인의 권리ㆍ의무 관계에 중대한 영향을 미치는 판단 또는 평가
아. 「교통안전법」 제2조제1호부터 제3호까지에 따른 교통수단, 교통시설, 교통체계의 주요한 작동 및 운영
자. 공공서비스 제공에 필요한 자격 확인 및 결정 또는 비용징수 등 국민에게 영향을 미치는 국가, 지방자치단체, 「공공기관의 운영에 관한 법률」 제4조에 따른 공공기관 등(이하 "국가기관등"이라 한다)의 의사결정
차. 「교육기본법」 제9조제1항에 따른 유아교육ㆍ초등교육 및 중등교육에서의 학생 평가
카. 그 밖에 사람의 생명ㆍ신체의 안전 및 기본권 보호에 중대한 영향을 미치는 영역으로서 대통령령으로 정하는 영역

계층:
node_id: 6cb61391-b8eb-5073-9b4c-39f5

## 3. 임베딩과 Qdrant 저장

새 컬렉션 `ai_basic_law_hierarchy_v1`을 사용합니다. 기존 컬렉션은 유지합니다.


In [4]:
embedding_client = get_embedding_model()
qdrant_client = get_qdrant_client()
COLLECTION_NAME = "ai_basic_law_hierarchy_v1"

def embedding_text(chunk):
    location = " ".join(
        value
        for value in [
            chunk["law_name"],
            chunk["article"],
            chunk["paragraph"],
            chunk["item"],
            chunk["subitem"],
            chunk["title"],
        ]
        if value
    )
    return f"{location}\n{chunk['content']}"

texts = [embedding_text(chunk) for chunk in chunks]
vectors = await embedding_client.aembed_documents(texts)

if qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.delete_collection(COLLECTION_NAME)

qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=len(vectors[0]),
        distance=Distance.COSINE,
    ),
)

BATCH_SIZE = 64
for start in range(0, len(chunks), BATCH_SIZE):
    chunk_batch = chunks[start:start + BATCH_SIZE]
    vector_batch = vectors[start:start + BATCH_SIZE]

    points = [
        PointStruct(
            id=chunk["id"],
            vector=vector,
            payload=chunk,
        )
        for chunk, vector in zip(chunk_batch, vector_batch)
    ]

    qdrant_client.upsert(
        collection_name=COLLECTION_NAME,
        points=points,
        wait=True,
    )

print(f"{len(chunks)}개 계층형 청크 저장 완료: {COLLECTION_NAME}")


414개 계층형 청크 저장 완료: ai_basic_law_hierarchy_v1


## 4. Dense 검색 확인


In [5]:
query = "고영향 인공지능이란 무엇인가요?"
query_vector = await embedding_client.aembed_query(query)

result = qdrant_client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=10,
)

for rank, point in enumerate(result.points, start=1):
    payload = point.payload
    print(
        rank,
        f"[{payload['chunk_type']}]",
        payload["article"],
        payload["paragraph"],
        payload["item"],
        payload["subitem"],
        payload["title"],
        f"score={point.score:.4f}",
    )
    print(payload["content"])
    print()


1 [item] 제34조 ① 6.  고영향 인공지능과 관련한 사업자의 책무 score=0.6065
6. 그 밖에 고영향 인공지능의 안전성ㆍ신뢰성 확보를 위하여 위원회에서 심의ㆍ의결된 사항

2 [paragraph] 제35조 ①   고영향 인공지능 영향평가 score=0.6060
① 인공지능사업자가 고영향 인공지능을 이용한 제품 또는 서비스를 제공하는 경우 사전에 사람의 기본권에 미치는 영향을 평가(이하 "영향평가"라 한다)하기 위하여 노력하여야 한다. 이 경우 영향평가에는 고영향 인공지능을 이용한 제품 또는 서비스의 성격을 고려하여 인공지능취약계층의 특성이 반영될 수 있도록 하여야 한다. <개정 2026.1.20>

3 [item] 제8조 ① 10.  위원회의 기능 score=0.6034
10. 고영향 인공지능 규율에 관한 사항

4 [article] 제35조    고영향 인공지능 영향평가 score=0.5976
제35조(고영향 인공지능 영향평가)

5 [item] 제2조 1항 4.  정의 score=0.5972
4. "고영향 인공지능"이란 사람의 생명, 신체의 안전 및 기본권에 중대한 영향을 미치거나 위험을 초래할 우려가 있는 인공지능시스템으로서 다음 각 목의 어느 하나의 영역에서 활용되는 것을 말한다.

6 [paragraph] 제33조 ①   고영향 인공지능의 확인 score=0.5889
① 인공지능사업자는 인공지능 또는 이를 이용한 제품ㆍ서비스를 제공하는 경우 그 인공지능이 고영향 인공지능에 해당하는지에 대하여 사전에 검토하여야 하며, 필요한 경우 과학기술정보통신부장관에게 고영향 인공지능에 해당하는지 여부의 확인을 요청할 수 있다.

7 [article] 제33조    고영향 인공지능의 확인 score=0.5850
제33조(고영향 인공지능의 확인)

8 [paragraph] 제34조 ①   고영향 인공지능과 관련한 사업자의 책무 score=0.5823
① 인공지능사업자는 고영향 인공지능 또는 이를 이용한 제품ㆍ서비스를 제공하는 경우 고영향 인공지능의 안

## 5. 검색 결과를 답변 Context로 확장

검색된 호 청크는 `context_content`를 사용하면 하위 목까지 포함됩니다. 필요하면 `parent_id`와 `article_id`로 상위 노드를 조회할 수 있습니다.


In [6]:
def context_from_results(points):
    contexts = []
    seen = set()

    for point in points:
        payload = point.payload
        node_id = payload["node_id"]
        if node_id in seen:
            continue
        seen.add(node_id)

        contexts.append({
            "article": payload["article"],
            "paragraph": payload["paragraph"],
            "item": payload["item"],
            "subitem": payload["subitem"],
            "content": payload.get("context_content") or payload["content"],
        })

    return contexts

contexts = context_from_results(result.points[:5])
for context in contexts:
    print(
        context["article"],
        context["paragraph"],
        context["item"],
        context["subitem"],
    )
    print(context["content"][:1500])
    print()


제34조 ① 6. 
6. 그 밖에 고영향 인공지능의 안전성ㆍ신뢰성 확보를 위하여 위원회에서 심의ㆍ의결된 사항

제35조 ①  
① 인공지능사업자가 고영향 인공지능을 이용한 제품 또는 서비스를 제공하는 경우 사전에 사람의 기본권에 미치는 영향을 평가(이하 "영향평가"라 한다)하기 위하여 노력하여야 한다. 이 경우 영향평가에는 고영향 인공지능을 이용한 제품 또는 서비스의 성격을 고려하여 인공지능취약계층의 특성이 반영될 수 있도록 하여야 한다. <개정 2026.1.20>

제8조 ① 10. 
10. 고영향 인공지능 규율에 관한 사항

제35조   
제35조(고영향 인공지능 영향평가)
① 인공지능사업자가 고영향 인공지능을 이용한 제품 또는 서비스를 제공하는 경우 사전에 사람의 기본권에 미치는 영향을 평가(이하 "영향평가"라 한다)하기 위하여 노력하여야 한다. 이 경우 영향평가에는 고영향 인공지능을 이용한 제품 또는 서비스의 성격을 고려하여 인공지능취약계층의 특성이 반영될 수 있도록 하여야 한다. <개정 2026.1.20>
② 국가기관등이 고영향 인공지능을 이용한 제품 또는 서비스를 이용하려는 경우에는 영향평가를 실시한 제품 또는 서비스를 우선적으로 고려하여야 한다.
③ 그 밖에 영향평가의 구체적인 내용ㆍ방법 등에 관하여 필요한 사항은 대통령령으로 정한다.

제2조 1항 4. 
4. "고영향 인공지능"이란 사람의 생명, 신체의 안전 및 기본권에 중대한 영향을 미치거나 위험을 초래할 우려가 있는 인공지능시스템으로서 다음 각 목의 어느 하나의 영역에서 활용되는 것을 말한다.
가. 「에너지법」 제2조제1호에 따른 에너지의 공급
나. 「먹는물관리법」 제3조제1호에 따른 먹는물의 생산 공정
다. 「보건의료기본법」 제3조제1호에 따른 보건의료의 제공 및 이용체계의 구축ㆍ운영
라. 「의료기기법」 제2조제1항에 따른 의료기기 및 「디지털의료제품법」 제2조제2호에 따른 디지털의료기기의 개발 및 이용
마. 「원자력시설 등의 방호 및 방사능 방재 대책법」 제2조제1항제1호에 따른 핵물질과

## 평가 노트북 연결

`ai_basic_law_rag_evaluation.ipynb`의 컬렉션 이름을 변경합니다.

```python
COLLECTION_NAME = "ai_basic_law_hierarchy_v1"
```

검색에는 `content`를 사용하고, 답변 Context에는 `context_content`를 사용합니다.
